# **Dataset Curation**
This notebook consists of the code for dataset curation of 2 subsets of the Reddit data used for data visualizations. Given that open-index/arctic has 15.7B comments/submissions from 2005-2026, smaller datasets will be created to use for the visualizations.
## **Subset 1**
 Submissions (not comments) will be considered from 5 popular subreddits from generalized categories.
* **Additional note** - subreddits weren't implemented until 2009, so data from 2010-2026 will be used to ensure that the subreddit field is populated in all of the entries.
### Categories
There are thousands of subreddits out there; they will be generalized/grouped as follows:
* News and Politics
* Technology and Science
* Business and Finance
* Education and Career
* Discussion and Support
* Media and Entertainment
* Lifestyle, Hobbies and Living
### Sampling Process
Random sampling to get 400 posts from 5 popular subreddits of each category for each year from 2010-2026. To ensure that some posts have at least some degree of engagement:
* 200 of the 400 posts will be required to have a score of at least 50
* The remaining 200 of the 400 posts will be selected randomly
## **Subset 2**
This subset is specifically for manufactured engagement detection in political subreddits. All posts in the 5 popular/top politics subreddits from April 2025-February 2026 will be included. Specifically:
* r/PoliticalDiscussion
* r/politics
* r/worldnews
* r/Conservative
* r/conspiracy


# Subset 1 Generation

In [ ]:
# imports
from datasets import load_dataset
import duckdb
import os
import time
import pandas as pd
import shutil

# duckdb initializations to read from huggingface datasets via httpfs
db_cnx = duckdb.connect()
db_cnx.execute("INSTALL httpfs;")
db_cnx.execute("LOAD httpfs;")

In [ ]:
# uncomment and add token
# # hugging face token to bypass 429 error
# db_cnx.execute("""
#     CREATE SECRET hf_token (
#         TYPE HUGGINGFACE,
#         TOKEN ''
#     );
# """)

In [ ]:
# mounting google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 5 popular subreddits from each category
target_subreddits = [
    # news and politics
    "politicaldiscussion", "politics", "worldnews", "conservative", "conspiracy",
    # technology and science
    "chatgpt", "askscience", "technology", "singularity", "machinelearning",
    # business and finance
    "wallstreetbets", "personalfinance", "investing", "fire", "cryptocurrency",
    # education and career
    "recruitinghell", "careerguidance", "jobs", "cscareerquestions", "antiwork",
    # discussion and support
    "askreddit", "unpopularopinion", "nostupidquestions", "amitheasshole", "offmychest",
    # media and entertainment
    "books", "popculturechat", "fauxmoi", "netflix", "television",
    # lifestyle, hobbies, and living
    "diy", "travel", "skincareaddiction", "lifeprotips", "fitness"
]

In [ ]:
# dropping table from previous run to restart after 404 error
db_cnx.execute("DROP TABLE IF EXISTS master_dataset;")

In [ ]:
# format target subreddits for sql
target_subreddits = [s.lower() for s in target_subreddits]
sub_list_str = ','.join([f"'{s}'" for s in target_subreddits])

# base directory for checkpoints on google drive
base_dir = '/content/drive/MyDrive/curated-reddit-data-general'

# loop monthly from jan 2020 to sep 2026
for year in range(2010, 2027):
    max_month = 9 if year == 2026 else 12
    for month in range(1, max_month + 1):
        part_filename = f"reddit_{year}_{month:02d}.parquet"
        part_path = os.path.join(base_dir, part_filename)

        # skip if this month's part is already saved in the google drive
        if os.path.exists(part_path):
            print(f"Skipping {year}-{month:02d} (already saved to Drive)...")
            continue


        print(f"Processing and saving submissions for {year}-{month:02d}...")

        month_query = f"""
            COPY (
                WITH filtered_data AS (
                    SELECT
                        *,
                        EXTRACT(YEAR FROM epoch_ms(CAST(created_utc AS BIGINT) * 1000)) AS post_year,
                        EXTRACT(MONTH FROM epoch_ms(CAST(created_utc AS BIGINT) * 1000)) AS post_month
                    FROM 'hf://datasets/open-index/arctic/data/submissions/{year}/{month:02d}/*.parquet'
                    WHERE LOWER(subreddit) IN ({sub_list_str})
                      AND selftext IS NOT NULL
                      AND selftext != ''
                      AND selftext NOT IN ('[removed]', '[deleted]')
                ),

                -- pool 1: high engagement (score >= 50)
                high_score_pool AS (
                    SELECT *,
                           ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY RANDOM()) as rn
                    FROM filtered_data
                    WHERE score >= 50
                ),
                sampled_high AS (
                    SELECT * EXCLUDE (rn) FROM high_score_pool WHERE rn <= 200
                ),

                -- pool 2: purely random sample (rest of the data)
                random_pool AS (
                    SELECT *,
                           ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY RANDOM()) as rn
                    FROM filtered_data
                ),
                sampled_random AS (
                    SELECT * EXCLUDE (rn) FROM random_pool WHERE rn <= 200
                )

                -- combine both pools for this month
                SELECT * FROM sampled_high
                UNION
                SELECT * FROM sampled_random
            ) TO '{part_path}' (FORMAT PARQUET, COMPRESSION SNAPPY);
        """

        try:
            db_cnx.execute(month_query)
            # adding pause to prevent 429 error
            time.sleep(30)
        except Exception as e:
            err_msg = str(e)
            # handling 404 missing folders
            if "404" in err_msg:
                print(f"Folder for {year}-{month:02d} not found on Hugging Face (404). Skipping")
                time.sleep(10)
                continue
            else:
                raise e

print("Dataset curation and checkpointing complete")

Skipping 2010-01 (already saved to Drive)...
Skipping 2010-02 (already saved to Drive)...
Skipping 2010-03 (already saved to Drive)...
Skipping 2010-04 (already saved to Drive)...
Skipping 2010-05 (already saved to Drive)...
Skipping 2010-06 (already saved to Drive)...
Skipping 2010-07 (already saved to Drive)...
Skipping 2010-08 (already saved to Drive)...
Skipping 2010-09 (already saved to Drive)...
Skipping 2010-10 (already saved to Drive)...
Skipping 2010-11 (already saved to Drive)...
Skipping 2010-12 (already saved to Drive)...
Skipping 2011-01 (already saved to Drive)...
Skipping 2011-02 (already saved to Drive)...
Skipping 2011-03 (already saved to Drive)...
Skipping 2011-04 (already saved to Drive)...
Skipping 2011-05 (already saved to Drive)...
Processing and saving submissions for 2011-06...
Folder for 2011-06 not found on Hugging Face (404). Skipping
Skipping 2011-07 (already saved to Drive)...
Skipping 2011-08 (already saved to Drive)...
Skipping 2011-09 (already saved to 

In [ ]:
# checking how many subreddits in a parquet - should be 5 * 7 = 35
raw_check = duckdb.sql("""
    SELECT COUNT(DISTINCT subreddit) AS total_subreddits
    FROM read_parquet('/content/drive/MyDrive/curated-reddit-data-general/reddit_2026_02.parquet');
""").df()

print(raw_check)

   total_subreddits
0                34


In [ ]:
# printing the subreddits in the parquet
unique_subs_df = duckdb.sql("""
    SELECT DISTINCT subreddit
    FROM read_parquet('/content/drive/MyDrive/curated-reddit-data-general/reddit_2026_02.parquet')
    WHERE subreddit IS NOT NULL
    ORDER BY subreddit ASC;
""").df()

# display the complete list
pd.set_option('display.max_rows', None)
display(unique_subs_df)

,subreddit
0,AmItheAsshole
1,ChatGPT
2,Conservative
3,CryptoCurrency
4,DIY
5,Fauxmoi
6,Fire
7,Fitness
8,LifeProTips
9,MachineLearning


In [ ]:
# combining individual monthly parts on google drive and merging into final master dataset
export_query = """
COPY (
    SELECT * FROM read_parquet('/content/drive/MyDrive/curated-reddit-data-general/reddit_*.parquet')
) TO '/content/drive/MyDrive/curated-reddit-data-general/combined-reddit-data.parquet'
(FORMAT PARQUET, COMPRESSION SNAPPY);
"""

# execute the final merge
db_cnx.execute(export_query)
print("Final master dataset merge complete")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final master dataset merge complete


In [ ]:
# outputting the size of the merged master parquet (2010-2026)
df_test = db_cnx.sql("""
    SELECT count(*) AS submissions
    FROM read_parquet('/content/drive/MyDrive/curated-reddit-data-general/combined-reddit-data.parquet');
""").df()
print(df_test)

   submissions
0      1154517


In [ ]:
stats_df = db_cnx.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(CASE WHEN selftext IS NOT NULL AND selftext != '' THEN 1 END) AS total_non_empty,
        COUNT(CASE WHEN selftext = '[removed]' THEN 1 END) AS removed_count,
        COUNT(CASE WHEN selftext = '[deleted]' THEN 1 END) AS deleted_count,
        COUNT(CASE WHEN selftext IS NOT NULL AND selftext != '' AND selftext NOT IN ('[removed]', '[deleted]') THEN 1 END) AS clean_text_count,
        ROUND(100.0 * COUNT(CASE WHEN selftext IS NOT NULL AND selftext != '' AND selftext NOT IN ('[removed]', '[deleted]') THEN 1 END) / COUNT(*), 2) AS pct_clean_text
    FROM read_parquet('/content/drive/MyDrive/curated-reddit-data-general/combined-reddit-data.parquet');
""").df()

display(stats_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,total_non_empty,removed_count,deleted_count,clean_text_count,pct_clean_text
0,1154517,1154517,0,0,1154517,100.0


# Subset 2 Generation

In [ ]:
# configure duckdb to leverage colab resources
db_cnx.execute("PRAGMA threads=8;")
db_cnx.execute("PRAGMA memory_limit='25GB';")

target_political_subreddits = ["politicaldiscussion", "politics", "worldnews", "conservative", "conspiracy"]
target_political_subreddits = [s.lower() for s in target_political_subreddits]
sub_list_str = ','.join([f"'{s}'" for s in target_political_subreddits])

base_dir = '/content/drive/MyDrive/curated-reddit-data-political'
os.makedirs(base_dir, exist_ok=True)

# target april 2025 - march 2026
for year in [2025, 2026]:
    start_month = 4 if year == 2025 else 1
    end_month = 3 if year == 2026 else 12

    for month in range(start_month, end_month + 1):

        sub_filename = f"submissions_{year}_{month:02d}.parquet"
        drive_sub_path = os.path.join(base_dir, sub_filename)
        local_sub_path = f"/content/{sub_filename}"

        if os.path.exists(drive_sub_path):
            print(f"Skipping submissions for {year}-{month:02d} (already in Drive)...")
        else:
            print(f"Processing submissions for {year}-{month:02d}...")

            sub_query = f"""
                COPY (
                    SELECT
                        id,
                        author,
                        created_utc,
                        subreddit,
                        score,
                        title,
                        selftext,
                        url,
                        num_comments,
                        author_flair_text,
                        link_flair_text
                    FROM 'hf://datasets/open-index/arctic/data/submissions/{year}/{month:02d}/*.parquet'
                    WHERE LOWER(subreddit) IN ({sub_list_str})
                      AND author NOT IN ('[deleted]')
                      AND (selftext IS NULL OR selftext NOT IN ('[removed]', '[deleted]'))
                ) TO '{local_sub_path}' (FORMAT PARQUET, COMPRESSION SNAPPY);
            """
            try:
                db_cnx.execute(sub_query)
                print("Moving submissions to Google Drive...")
                shutil.move(local_sub_path, drive_sub_path)
                time.sleep(10)
            except Exception as e:
                if "404" in str(e):
                    print(f"Submissions 404 for {year}-{month:02d}. Skipping.")
                    time.sleep(5)
                else:
                    raise e

print("Political submissions dataset curation complete")

Skipping submissions for 2025-04 (already in Drive)...
Skipping submissions for 2025-05 (already in Drive)...
Skipping submissions for 2025-06 (already in Drive)...
Processing submissions for 2025-07...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Moving submissions to Google Drive...
Processing submissions for 2025-08...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Moving submissions to Google Drive...
Processing submissions for 2025-09...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (7 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Moving submissions to Google Drive...
Processing submissions for 2026-02...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Moving submissions to Google Drive...
Processing submissions for 2026-03...
Submissions 404 for 2026-03. Skipping.
Political submissions dataset curation complete


In [ ]:
# querying a random submission from a generated parquet
file_path = '/content/drive/MyDrive/curated-reddit-data-political/submissions_2025_04.parquet'

# convert a random row to a pandas df and display it
random_submission = db_cnx.execute(f"SELECT * FROM '{file_path}' USING SAMPLE 1 ROWS").df()
display(random_submission)

# printing the specific columns
print("\n--- Details ---")
print("Subreddit:", random_submission['subreddit'].iloc[0])
print("Author:", random_submission['author'].iloc[0])
print("Title:", random_submission['title'].iloc[0])
print("URL:", random_submission['url'].iloc[0])

,id,author,created_utc,subreddit,score,title,selftext,url,num_comments,author_flair_text,link_flair_text
0,1jqb42b,Jaded-Bookkeeper-807,1743661124,worldnews,5914,Starlink rival Eutelsat rolls out wifi on Air ...,,https://24newshd.tv/03-Apr-2025/starlink-rival...,85,None,None



--- Details ---
Subreddit: worldnews
Author: Jaded-Bookkeeper-807
Title: Starlink rival Eutelsat rolls out wifi on Air Canada planes
URL: https://24newshd.tv/03-Apr-2025/starlink-rival-eutelsat-rolls-out-wifi-on-air-canada-planes


In [ ]:
# querying for all submissions by a user
file_path = '/content/drive/MyDrive/curated-reddit-data-political/submissions_2025_04.parquet'

# username to check
target_author = 'Jaded-Bookkeeper-807'

# querying all rows where user matches (ignoring case sensitivity)
query = f"""
    SELECT *
    FROM '{file_path}'
    WHERE LOWER(author) = LOWER('{target_author}')
    ORDER BY created_utc DESC
"""

# run query and create df
user_submissions = db_cnx.execute(query).df()

# print total; display their posts
print(f"Found {len(user_submissions)} submissions by {target_author} in this month.")
display(user_submissions)

Found 68 submissions by Jaded-Bookkeeper-807 in this month.


,id,author,created_utc,subreddit,score,title,selftext,url,num_comments,author_flair_text,link_flair_text
0,1kb8hzg,Jaded-Bookkeeper-807,1745989269,worldnews,34,"Cardinal Tagle, the 'Asian Francis,' is papal ...",,https://www.gmanetwork.com/news/topstories/nat...,11,None,None
1,1kb897m,Jaded-Bookkeeper-807,1745988340,politics,0,VP Vance accused Europe of stifling free speec...,,https://www.deseret.com/u-s-world/2025/04/27/w...,16,None,None
2,1kb6f3j,Jaded-Bookkeeper-807,1745981995,politics,1,Trump said that Putin would have occupied the ...,,https://unn.ua/en/amp/trump-said-that-putin-wo...,1,None,Non-approved domain
3,1kb6bi9,Jaded-Bookkeeper-807,1745981670,worldnews,57,China adopts law to protect private companies ...,,https://theedgemalaysia.com/node/753506,2,None,None
4,1kb5sw4,Jaded-Bookkeeper-807,1745980065,worldnews,377,Papal front-runner: Cardinal Ambongo could be ...,,https://www.ncronline.org/vatican/vatican-news...,87,None,None
...,...,...,...,...,...,...,...,...,...,...,...
63,1jp2x6h,Jaded-Bookkeeper-807,1743532088,politics,4877,Trump makes history by pardoning a corporation,,https://thehill.com/opinion/criminal-justice/5...,611,None,None
64,1jp2qf5,Jaded-Bookkeeper-807,1743531645,politics,1,Opinion: Trump makes history by pardoning a co...,,https://www.msn.com/en-us/news/politics/opinio...,1,None,Rehosted Content
65,1jop5vn,Jaded-Bookkeeper-807,1743489417,worldnews,79,Vietnam slashes duties on range of imports to ...,,https://www.channelnewsasia.com/asia/vietnam-s...,5,None,None
66,1jop1c7,Jaded-Bookkeeper-807,1743488873,politics,327,A Senate vote to reverse Trump’s tariffs on Ca...,,https://apnews.com/article/tariffs-canada-cong...,64,None,None


In [ ]:
# combining the parquets for subset 2
db_cnx.execute("PRAGMA threads=8;")
db_cnx.execute("PRAGMA memory_limit='25GB';")

base_dir = '/content/drive/MyDrive/curated-reddit-data-political'
input_pattern = os.path.join(base_dir, 'submissions_*.parquet')
output_file = os.path.join(base_dir, 'combined-reddit-data-political.parquet')

print(f"Merging all files matching: {input_pattern}")
print(f"Outputting to: {output_file}")

# reading all matched files and writing them to the new output file parquet
merge_query = f"""
    COPY (
        SELECT *
        FROM '{input_pattern}'
        ORDER BY created_utc ASC
    ) TO '{output_file}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""

try:
    db_cnx.execute(merge_query)
    print("Merge complete")
except Exception as e:
    print(f"An error occurred during merging: {e}")

Merging all files matching: /content/drive/MyDrive/curated-reddit-data-political/submissions_*.parquet
Outputting to: /content/drive/MyDrive/curated-reddit-data-political/combined-reddit-data-political.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merge complete


In [ ]:
# outputting the size of the merged political data
df_test = db_cnx.sql("""
    SELECT count(*) AS submissions
    FROM read_parquet('/content/drive/MyDrive/curated-reddit-data-political/combined-reddit-data-political.parquet');
""").df()
print(df_test)

   submissions
0       361184
